<a href="https://colab.research.google.com/github/jheyson1984/ml-s8-clasificacion/blob/main/ML_S8_Clasificacion_MoralesJheyson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
from sklearn.datasets import load_breast_cancer
import pandas as pd

data = load_breast_cancer(as_frame=True)
X = data.data.copy()
y = (data.target == 0).astype(int)

print(X.shape)
print(y.value_counts())

(569, 30)
target
0    357
1    212
Name: count, dtype: int64


In [15]:
X.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [16]:
y.value_counts(normalize=True)

,proportion
target,
0,0.627417
1,0.372583


**¿Cuántas observaciones y variables contiene el conjunto de datos?**
569 observaciones y 30 variables.

**¿La variable objetivo está balanceada?**
No. El 63 % son benignos y el 37 % malignos; está desbalanceada, casi 2 a 1. Es un desbalance moderado.

**¿Por qué es importante definir con claridad qué representa la clase positiva?**
Es importante hacerle seguimiento a los positivos. Las métricas como el recall (sensibilidad) se calculan sobre la clase positiva: de todos los malignos reales, cuántos detectó el modelo. Si alguien confunde cuál es el grupo positivo, no se le haría seguimiento a un caso con resultado maligno

In [17]:
print(data.DESCR)

.. _breast_cancer_dataset:

Breast cancer wisconsin (diagnostic) dataset
--------------------------------------------

**Data Set Characteristics:**

:Number of Instances: 569

:Number of Attributes: 30 numeric, predictive attributes and the class

:Attribute Information:
    - radius (mean of distances from center to points on the perimeter)
    - texture (standard deviation of gray-scale values)
    - perimeter
    - area
    - smoothness (local variation in radius lengths)
    - compactness (perimeter^2 / area - 1.0)
    - concavity (severity of concave portions of the contour)
    - concave points (number of concave portions of the contour)
    - symmetry
    - fractal dimension ("coastline approximation" - 1)

    The mean, standard error, and "worst" or largest (mean of the three
    worst/largest values) of these features were computed for each image,
    resulting in 30 features.  For instance, field 0 is Mean Radius, field
    10 is Radius SE, field 20 is Worst Radius.

    - 

In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(455, 30) (114, 30)
target
0    0.626374
1    0.373626
Name: proportion, dtype: float64
target
0    0.631579
1    0.368421
Name: proportion, dtype: float64


**¿Qué función cumple stratify=y?**
Reparte los malignos en la misma proporción en entrenamiento (37,4 %) y en prueba (36,8 %), igual que en la base completa (37,3 %). Si no se usara, los casos tendrían una opción desigual de quedar en los dos grupos, y con pocos malignos en la prueba no se podría evaluar bien cuántos detecta el modelo.

In [19]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)

def evaluar_modelo(nombre, modelo, X_eval, y_eval):
    pred = modelo.predict(X_eval)

    resultados = {
        'Modelo': nombre,
        'Accuracy': accuracy_score(y_eval, pred),
        'Precision': precision_score(y_eval, pred),
        'Recall': recall_score(y_eval, pred),
        'F1': f1_score(y_eval, pred)
    }

    print(nombre)
    print(confusion_matrix(y_eval, pred))
    return resultados

In [20]:
from sklearn.tree import DecisionTreeClassifier

modelo_dt = DecisionTreeClassifier(max_depth=4, random_state=42)
modelo_dt.fit(X_train, y_train)

res_dt = evaluar_modelo("Decision Tree", modelo_dt, X_test, y_test)
res_dt

Decision Tree
[[70  2]
 [ 8 34]]


{'Modelo': 'Decision Tree',
 'Accuracy': 0.9122807017543859,
 'Precision': 0.9444444444444444,
 'Recall': 0.8095238095238095,
 'F1': 0.8717948717948718}

**¿Qué ventaja tiene limitar la profundidad del árbol?**
Lo hace más fácil de explicar y aplicar, como un protocolo de pocos pasos. Es menos exacto con los datos de entrenamiento, pero eso lo hace más aplicable en otros entornos, porque evita que el modelo memorice las particularidades de los datos (sobreajuste).

**¿Qué errores se observan en la matriz de confusión?**
El modelo no detectó 8 malignos (falsos negativos) y marcó 2 benignos como malignos (falsos positivos). Los 8 malignos no detectados son el error más grave, porque quedarían sin seguimiento.

In [21]:
from sklearn.ensemble import RandomForestClassifier

modelo_rf = RandomForestClassifier(n_estimators=100, random_state=42)
modelo_rf.fit(X_train, y_train)

res_rf = evaluar_modelo("Random Forest", modelo_rf, X_test, y_test)
res_rf

Random Forest
[[72  0]
 [ 3 39]]


{'Modelo': 'Random Forest',
 'Accuracy': 0.9736842105263158,
 'Precision': 1.0,
 'Recall': 0.9285714285714286,
 'F1': 0.9629629629629629}

**¿Qué cambia al pasar de un árbol individual a un conjunto de árboles?**
Cada árbol tiene criterios diferentes para clasificar, y la decisión se toma por mayoría de votos. Al igual que en un grupo de profesionales, los errores de uno los corrigen los demás. A cambio, el modelo es más difícil de explicar que un solo árbol.

**¿Mejora alguna métrica? ¿Cuál y cuánto?**
Mejoran todas. Acertó el 100 % de los benignos y solo dejó pasar 3 malignos, frente a los 8 del árbol de decisión. El recall sube de 0,81 a 0,93, la precision de 0,94 a 1,00, la accuracy de 0,91 a 0,97 y el F1 de 0,87 a 0,96.

In [22]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

modelo_svm = make_pipeline(
    StandardScaler(),
    SVC(kernel='rbf', C=1.0, gamma='scale')
)
modelo_svm.fit(X_train, y_train)

res_svm = evaluar_modelo("SVM", modelo_svm, X_test, y_test)
res_svm

SVM
[[72  0]
 [ 3 39]]


{'Modelo': 'SVM',
 'Accuracy': 0.9736842105263158,
 'Precision': 1.0,
 'Recall': 0.9285714285714286,
 'F1': 0.9629629629629629}

**¿Por qué en este modelo usamos StandardScaler?**
Porque SVM decide midiendo distancias, y sin escalar pesaría más el área del núcleo solo por tener números más grandes que la suavidad. El escalamiento pone todas las variables en la misma regla, como pasar casos a tasas para comparar municipios.

**¿Qué resultado obtiene SVM frente a los modelos basados en árboles?**
Salió el mismo resultado que Random Forest (3 malignos no detectados y 0 falsas alarmas) y mejor que el árbol de decisión, que dejó pasar 8 malignos.

In [23]:
from sklearn.neighbors import KNeighborsClassifier

modelo_knn = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=5)
)
modelo_knn.fit(X_train, y_train)

res_knn = evaluar_modelo("K-NN", modelo_knn, X_test, y_test)
res_knn

K-NN
[[71  1]
 [ 4 38]]


{'Modelo': 'K-NN',
 'Accuracy': 0.956140350877193,
 'Precision': 0.9743589743589743,
 'Recall': 0.9047619047619048,
 'F1': 0.9382716049382716}

**¿Por qué el escalamiento es especialmente importante cuando se calculan distancias?**
Porque K-NN busca los casos más parecidos midiendo distancias. Sin escalar, la variable con números más grandes (el área del núcleo) pesaría más solo por su tamaño, y no por ser más importante.

**¿Qué representa k=5 en este algoritmo?**
Es el número de vecinos que el modelo consulta. Para clasificar una muestra nueva busca las 5 muestras de entrenamiento más parecidas y decide por mayoría: si 3 o más son malignas, la clasifica como maligna. K-NN tuvo 1 falsa alarma y dejó pasar 4 malignos.

In [24]:
from sklearn.naive_bayes import GaussianNB

modelo_nb = GaussianNB()
modelo_nb.fit(X_train, y_train)

res_nb = evaluar_modelo("Naive Bayes", modelo_nb, X_test, y_test)
res_nb

Naive Bayes
[[72  0]
 [ 7 35]]


{'Modelo': 'Naive Bayes',
 'Accuracy': 0.9385964912280702,
 'Precision': 1.0,
 'Recall': 0.8333333333333334,
 'F1': 0.9090909090909091}

**¿Qué supuesto simplificador utiliza Naive Bayes respecto a las variables?**
Supone que las variables son independientes entre sí: cada una suma por separado, como los criterios de una definición de caso, sin tener en cuenta que algunas van juntas (por ejemplo, el área y el perímetro del núcleo).

In [25]:
tabla = pd.DataFrame([res_dt, res_rf, res_svm, res_knn, res_nb])
tabla.round(3)

,Modelo,Accuracy,Precision,Recall,F1
0,Decision Tree,0.912,0.944,0.810,0.872
1,Random Forest,0.974,1.000,0.929,0.963
2,SVM,0.974,1.000,0.929,0.963
3,K-NN,0.956,0.974,0.905,0.938
4,Naive Bayes,0.939,1.000,0.833,0.909


## Ejercicio 7. Comparación y selección del modelo

| Modelo | Accuracy | Precision | Recall | F1 | Observación |
|---|---|---|---|---|---|
| Decision Tree | 0,912 | 0,944 | 0,810 | 0,872 | Dejó pasar 8 malignos; es el más fácil de explicar |
| Random Forest | 0,974 | 1,000 | 0,929 | 0,963 | 3 malignos no detectados, 0 falsas alarmas; sin escalamiento |
| SVM | 0,974 | 1,000 | 0,929 | 0,963 | Igual a Random Forest; necesita escalamiento |
| K-NN | 0,956 | 0,974 | 0,905 | 0,938 | 4 malignos no detectados y 1 falsa alarma |
| Naive Bayes | 0,939 | 1,000 | 0,833 | 0,909 | Sin falsas alarmas, pero dejó pasar 7 malignos |

**7.** Random Forest y SVM tuvieron un empate en la mayor accuracy (0,974).

**8.** Random Forest y SVM tuvieron un empate en el mayor recall para la clase maligna (0,929).

**9.** Random Forest y SVM empataron con el menor número de falsos negativos: 3 malignos no detectados cada uno. Además no tuvieron falsos positivos, por eso su precision es del 100 %.

**10.** Sí coincide, porque los modelos con mayor accuracy también tienen el mayor recall. Se necesita garantizar seguimiento a los verdaderos positivos (malignos); a pesar de que RF y SVM dejan pasar 3 casos, estos se detectarían de otra forma dentro del proceso clínico.

**11.** Priorizaría el recall (sensibilidad), porque mide cuántos de los malignos reales detecta el modelo. La accuracy puede verse alta aunque se escapen malignos, porque también cuenta los aciertos en benignos.

**12.** El Decision Tree, porque con máximo 4 preguntas seguidas se puede dibujar y explicar como un protocolo de pocos pasos.

**13.** SVM y K-NN necesitan escalamiento porque clasifican midiendo distancias entre muestras. Los árboles y Naive Bayes no lo necesitan: el árbol pregunta sobre una variable a la vez, sin comparar unas con otras. Por eso Random Forest, sin escalar, obtuvo el mismo resultado que SVM escalado.

**14.** Selecciono Random Forest. Tiene el mayor recall (0,929) y una precision del 100 %, empatado con SVM. Frente a SVM, no necesita escalamiento y permite ver qué variables pesaron más en la decisión, lo que facilita explicarle el resultado a un equipo clínico. Este modelo es un ejercicio académico y no reemplaza el diagnóstico médico.